# SFT2D — core tour

Build a grid, set up the flows, integrate, and check the diagnostics on the
pole-to-pole finite-volume mesh. The solver `evolve` does Strang splitting
internally: advection with SSPRK(3,3) sub-cycled at its CFL, diffusion with RKL2
super-time-stepping. There is no boundary condition to apply — the poles are
ordinary finite-volume cells.

In [ ]:
# Make `sft2d` importable when running from docs/notebooks/ without pip-installing.
import sys, pathlib
try:
    import sft2d
except ModuleNotFoundError:
    sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sft2d as sft
from sft2d import (create_grid, total_flux, initialize_field, evolve,
                   meridional_flow, meridional_flow_latitude, differential_rotation,
                   calculate_usflx, calculate_net_flux, calculate_dm,
                   calculate_polar_field, calculate_polar_flux, plot_bfly, plot_mag)
print("sft2d", sft.__version__)

## Grid

`n_theta` counts colatitude cells pole to pole, *including* the two polar caps,
so `create_grid(91, 180)` gives 2° cells spanning the full sphere. The cell
areas sum to $4\pi R^2$ exactly.

In [ ]:
grid = create_grid(91, 180)
lat = np.rad2deg(np.pi/2 - grid['colatitude'])
print(f"latitude range : {lat.min():+.1f} .. {lat.max():+.1f} deg")
print(f"cells          : {grid['n_theta']} x {grid['n_phi']}, dtheta = {np.rad2deg(grid['dtheta']):.2f} deg")
print(f"area closure   : {float(np.sum(grid['area']))*grid['n_phi']/(4*np.pi*sft.R_SUN_M**2):.15f}")

## Flows — and the meridional-flow sign convention

`meridional_flow(grid, peak_speed)` returns the **colatitude** velocity
`u_theta`, which is what the solver advects with. **Positive `peak_speed` is
poleward** (the physical case). Because `+theta` points southward everywhere, a
poleward `u_theta` is *negative in the north, positive in the south* — it looks
upside-down only because it is the colatitude component.

For an intuitive picture use `meridional_flow_latitude` (northward-positive
`v_lat = -u_theta`): a poleward flow is then `+` in the north and `-` in the
south, matching meridional-flow figures in the literature. A **negative**
`peak_speed` is equatorward and will prevent polar reversal.

In [ ]:
v0 = 15.0                                     # positive = poleward
mf   = meridional_flow(grid, peak_speed=v0)   # u_theta [m/s]  -> feed to the solver
vlat = meridional_flow_latitude(grid, peak_speed=v0)  # v_lat [m/s] -> for plotting
dr   = differential_rotation(grid, rotation='solar', frame='carrington')  # Omega [rad/s]

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(lat, mf[:, 0], 'C0', label=r'$u_\theta$ (solver)')
ax[0].plot(lat, vlat[:, 0], 'C3', label=r'$v_{lat}$ (poleward +)')
ax[0].axhline(0, color='k', lw=0.5); ax[0].axvline(0, color='k', lw=0.5)
ax[0].set_xlabel('latitude [deg]'); ax[0].set_ylabel('m/s'); ax[0].legend()
ax[0].set_title(f'meridional flow, peak_speed={v0:+g} (poleward)')
ax[1].plot(lat, dr[:, 0]*1e6, 'k'); ax[1].set_xlabel('latitude [deg]')
ax[1].set_ylabel(r'$\Omega$ [$10^{-6}$ rad/s]'); ax[1].set_title('differential rotation')
fig.tight_layout()

## Initial condition

In [ ]:
field0 = initialize_field(grid, 'dipole') * 3.0   # amplitude 3 G axisymmetric dipole
eta = 2.5e8                                        # m^2/s = 250 km^2/s
plot_mag(field0, grid, bmax=3)

## Integrate

`return_stats=True` reports the chosen stage / sub-cycle counts. Compare
`rkl2_stages` with `explicit_diffusion_steps_avoided` to see what
super-time-stepping buys.

In [ ]:
bfly, days = [], []
class Recorder:
    def record(self, day, B):
        if day % 5 == 0:
            days.append(day); bfly.append(B.mean(axis=1).copy())

B, stats = evolve(field0, grid, mf, dr, eta, num_days=365,
                  recorder=Recorder(), return_stats=True)
for k, v in stats.items():
    print(f"{k:38s} {v:.4g}")

## Conservation check

The transport operators are conservative by construction, so with no source term
the signed flux must not move.

In [ ]:
f0, f1 = calculate_net_flux(field0, grid), calculate_net_flux(B, grid)
u0 = calculate_usflx(field0, grid)
print(f"net flux   {f0:+.6e} -> {f1:+.6e} Mx")
print(f"drift      {abs(f1-f0)/u0:.3e}  (relative to total unsigned flux)")

## Diagnostics

In [ ]:
pn, ps = calculate_polar_field(B, grid, pol_cap_extent_deg=20)   # poleward of +/-70 deg
fn, fs = calculate_polar_flux(B, grid, pol_cap_extent_deg=20)
print(f"polar field  N {pn:+.3f} G      S {ps:+.3f} G")
print(f"polar flux   N {fn:+.4e} Mx  S {fs:+.4e} Mx")
print(f"axial dipole {calculate_dm(B, grid):+.4f} G")
print(f"unsigned     {calculate_usflx(B, grid):.4e} Mx")

In [ ]:
plot_bfly(np.array(bfly), grid, bmax=5)
plot_mag(B, grid, bmax=10)